# **Incorporación de SENTIMENT**

In [19]:
import time, requests, pandas as pd, numpy as np, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [20]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

model_name = "ProsusAI/finbert"
tok = AutoTokenizer.from_pretrained(model_name)
mdl = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
mdl.eval()


device: cuda


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [21]:
GDELT_DOC = "https://api.gdeltproject.org/api/v2/doc/doc"

def ymdhms(ts):
    return pd.Timestamp(ts).strftime("%Y%m%d%H%M%S")


In [22]:
def get_daily_sentiment(day, tries=4):
    start = pd.Timestamp(day)
    end = start + pd.Timedelta(days=1)

    q = '(AAPL OR "Apple stock" OR "Apple shares") sourcelang:eng'
    params = {
        "query": q,
        "mode": "artlist",
        "format": "json",
        "maxrecords": 250,
        "startdatetime": ymdhms(start),
        "enddatetime": ymdhms(end),
        "sort": "datedesc",
    }

    for k in range(tries):
        r = requests.get(GDELT_DOC, params=params, timeout=30)
        ct = (r.headers.get("content-type") or "").lower()

        if r.status_code == 200 and "json" in ct:
            j = r.json()
            arts = j.get("articles", [])
            if not arts:
                return None

            df = pd.DataFrame([{
                "seendate": a.get("seendate"),
                "title": a.get("title"),
                "url": a.get("url"),
            } for a in arts])

            df["dt"] = pd.to_datetime(df["seendate"], errors="coerce", utc=True)
            df["date"] = df["dt"].dt.date
            df = df.dropna(subset=["title","date","url"]).drop_duplicates(subset=["url"])
            df = df[df["title"].str.contains("Apple|AAPL", case=False, na=False)]

            if df.empty:
                return None

            texts = df["title"].tolist()
            enc = tok(texts, padding=True, truncation=True, max_length=64, return_tensors="pt")
            enc = {k: v.to(device) for k, v in enc.items()}  # ✅ evita CPU/GPU mismatch

            with torch.no_grad():
                probs = torch.softmax(mdl(**enc).logits, dim=-1).detach().cpu().numpy()

            scores = probs[:, 2] - probs[:, 0]  # P(pos) - P(neg)

            return {
                "date": df["date"].iloc[0],
                "sent_mean": float(scores.mean()),
                "sent_std": float(scores.std()) if len(scores) > 1 else 0.0,
                "sent_volume": int(len(scores)),
            }

        time.sleep(5 * (2 ** k))  # backoff si HTML/errores

    return None


In [23]:
row = get_daily_sentiment("2024-01-10")
print(row)


{'date': datetime.date(2024, 1, 10), 'sent_mean': 0.18508410453796387, 'sent_std': 0.4976111054420471, 'sent_volume': 14}


In [24]:
def run_range(start_date, end_date, sleep_s=3):
    results = []
    days = pd.date_range(start_date, end_date, inclusive="left")
    for d in days:
        print(d.date())
        row = get_daily_sentiment(d)
        if row:
            results.append(row)
        time.sleep(sleep_s)
    return pd.DataFrame(results)

daily_df = run_range("2024-01-01", "2024-02-01", sleep_s=3)
print(daily_df.shape)
print(daily_df.head())


2024-01-01
2024-01-02
2024-01-03
2024-01-04
2024-01-05
2024-01-06
2024-01-07
2024-01-08
2024-01-09
2024-01-10
2024-01-11
2024-01-12
2024-01-13
2024-01-14
2024-01-15
2024-01-16
2024-01-17
2024-01-18
2024-01-19
2024-01-20
2024-01-21
2024-01-22
2024-01-23
2024-01-24
2024-01-25
2024-01-26
2024-01-27
2024-01-28
2024-01-29
2024-01-30
2024-01-31
(29, 4)
         date  sent_mean  sent_std  sent_volume
0  2024-01-03   0.183025  0.433253           33
1  2024-01-03   0.191582  0.471204           41
2  2024-01-05   0.016481  0.375798           38
3  2024-01-05   0.111521  0.526958           24
4  2024-01-06   0.595997  0.344159            9


In [25]:
daily_df["date"] = pd.to_datetime(daily_df["date"]).dt.date

daily_df = (
    daily_df.groupby("date", as_index=False)
    .agg({"sent_mean":"mean", "sent_std":"mean", "sent_volume":"sum"})
    .sort_values("date")
    .reset_index(drop=True)
)

print(daily_df)


          date  sent_mean  sent_std  sent_volume
0   2024-01-03   0.187303  0.452229           74
1   2024-01-05   0.064001  0.451378           62
2   2024-01-06   0.595997  0.344159            9
3   2024-01-07   0.371536  0.305173           14
4   2024-01-09   0.443894  0.563348           37
5   2024-01-10   0.185084  0.497611           14
6   2024-01-11   0.008421  0.633809           39
7   2024-01-13  -0.177014  0.532023           60
8   2024-01-14   0.568833  0.502242            8
9   2024-01-15   0.021819  0.537251           12
10  2024-01-17   0.302627  0.392281           29
11  2024-01-18   0.146025  0.523141           80
12  2024-01-20   0.412844  0.534660           16
13  2024-01-21   0.665421  0.499893            6
14  2024-01-23   0.486406  0.440916           17
15  2024-01-25   0.196389  0.524678           43
16  2024-01-27   0.268750  0.719243           12
17  2024-01-28   0.421566  0.469453            4
18  2024-01-29   0.482947  0.352562           13
19  2024-01-30  -0.0

In [26]:
daily_df.to_csv("AAPL_sentiment_daily.csv", index=False)
print("saved:", daily_df.shape)


saved: (21, 4)


In [27]:
daily_df = daily_df.sort_values("date").reset_index(drop=True)

# rolling means (usan SOLO pasado, no futuro)
daily_df["sent_mean_3d"] = daily_df["sent_mean"].rolling(3, min_periods=1).mean()
daily_df["sent_mean_5d"] = daily_df["sent_mean"].rolling(5, min_periods=1).mean()

# momentum (cambio respecto al día anterior)
daily_df["sent_momentum"] = daily_df["sent_mean"] - daily_df["sent_mean"].shift(1)

print(daily_df.head(10))
print("\nNaNs por columna:")
print(daily_df.isna().sum())


         date  sent_mean  sent_std  sent_volume  sent_mean_3d  sent_mean_5d  \
0  2024-01-03   0.187303  0.452229           74      0.187303      0.187303   
1  2024-01-05   0.064001  0.451378           62      0.125652      0.125652   
2  2024-01-06   0.595997  0.344159            9      0.282434      0.282434   
3  2024-01-07   0.371536  0.305173           14      0.343845      0.304710   
4  2024-01-09   0.443894  0.563348           37      0.470476      0.332546   
5  2024-01-10   0.185084  0.497611           14      0.333505      0.332103   
6  2024-01-11   0.008421  0.633809           39      0.212466      0.320987   
7  2024-01-13  -0.177014  0.532023           60      0.005497      0.166384   
8  2024-01-14   0.568833  0.502242            8      0.133413      0.205844   
9  2024-01-15   0.021819  0.537251           12      0.137879      0.121429   

   sent_momentum  
0            NaN  
1      -0.123302  
2       0.531996  
3      -0.224461  
4       0.072358  
5      -0.258810

# A PARTIR DE AQUÍ ESTA LA SOLUCIÓN

In [30]:
import os
import pandas as pd
import time

RAW_OUT = "AAPL_news_raw.csv"

def fetch_block(start_date, end_date):
    q = '(AAPL OR "Apple stock" OR "Apple shares") sourcelang:eng'
    params = {
        "query": q,
        "mode": "artlist",
        "format": "json",
        "maxrecords": 250,
        "startdatetime": ymdhms(start_date),
        "enddatetime": ymdhms(end_date),
        "sort": "datedesc",
    }
    r = requests.get(GDELT_DOC, params=params, timeout=30)
    ct = (r.headers.get("content-type") or "").lower()
    if "json" not in ct or r.status_code != 200:
        return pd.DataFrame()

    arts = r.json().get("articles", [])
    df = pd.DataFrame([{
        "seendate": a.get("seendate"),
        "title": a.get("title"),
        "url": a.get("url")
    } for a in arts])
    return df

def download_weeks(start_date, end_date, sleep_s=2):
    all_parts = []
    cur = pd.Timestamp(start_date)
    end = pd.Timestamp(end_date)

    while cur < end:
        nxt = min(cur + pd.Timedelta(days=7), end)
        df = fetch_block(cur, nxt)

        # si una semana también satura, bajamos a bloques de 2 días dentro
        if len(df) >= 250:
            sub = cur
            subs = []
            while sub < nxt:
                sub_nxt = min(sub + pd.Timedelta(days=2), nxt)
                subs.append(fetch_block(sub, sub_nxt))
                sub = sub_nxt
                time.sleep(sleep_s)
            df = pd.concat(subs, ignore_index=True)

        all_parts.append(df)
        print(cur.date(), "->", nxt.date(), "rows:", len(df))
        cur = nxt
        time.sleep(sleep_s)

    raw = pd.concat(all_parts, ignore_index=True).dropna(subset=["title","url"]).drop_duplicates(subset=["url"])
    raw.to_csv(RAW_OUT, index=False)
    print("SAVED RAW:", raw.shape)

# PRUEBA: descarga Feb 2024 completo en semanas
download_weeks("2024-02-01", "2024-03-01", sleep_s=2)


2024-02-01 -> 2024-02-08 rows: 262
2024-02-08 -> 2024-02-15 rows: 175
2024-02-15 -> 2024-02-22 rows: 222
2024-02-22 -> 2024-02-29 rows: 207
2024-02-29 -> 2024-03-01 rows: 22
SAVED RAW: (880, 3)


In [31]:
import pandas as pd, numpy as np, torch

raw = pd.read_csv("AAPL_news_raw.csv").dropna(subset=["title","seendate","url"]).drop_duplicates(subset=["url"])
raw["dt"] = pd.to_datetime(raw["seendate"], errors="coerce", utc=True)
raw["date"] = raw["dt"].dt.date
raw = raw.dropna(subset=["date"]).reset_index(drop=True)

print("raw:", raw.shape, "min/max:", raw["date"].min(), raw["date"].max())

# FinBERT batches (GPU)
B = 64
scores = []

for i in range(0, len(raw), B):
    texts = raw["title"].iloc[i:i+B].tolist()
    enc = tok(texts, padding=True, truncation=True, max_length=64, return_tensors="pt")
    enc = {k: v.to(device) for k, v in enc.items()}
    with torch.no_grad():
        probs = torch.softmax(mdl(**enc).logits, dim=-1).detach().cpu().numpy()
    scores.append(probs[:,2] - probs[:,0])

raw["sent_score"] = np.concatenate(scores)

daily_df = raw.groupby("date").agg(
    sent_mean=("sent_score","mean"),
    sent_std=("sent_score","std"),
    sent_volume=("sent_score","size")
).reset_index()

daily_df["sent_std"] = daily_df["sent_std"].fillna(0.0)
daily_df = daily_df.sort_values("date").reset_index(drop=True)

daily_df.to_csv("AAPL_sentiment_daily.csv", index=False)

print("daily:", daily_df.shape)
print(daily_df.head())
print(daily_df.tail())


raw: (880, 5) min/max: 2024-02-01 2024-02-29
daily: (29, 4)
         date  sent_mean  sent_std  sent_volume
0  2024-02-01   0.004159  0.593859           50
1  2024-02-02   0.209864  0.518080           92
2  2024-02-03   0.268344  0.719226           18
3  2024-02-04   0.658170  0.354939           16
4  2024-02-05   0.283775  0.574816           35
          date  sent_mean  sent_std  sent_volume
24  2024-02-25   0.482201  0.401846           19
25  2024-02-26   0.473835  0.434726           26
26  2024-02-27   0.459953  0.462964           27
27  2024-02-28   0.385879  0.527119           75
28  2024-02-29   0.376918  0.457624           22


In [36]:
import requests, pandas as pd

def debug_block(start_date, end_date):
    q = '(AAPL OR "Apple stock" OR "Apple shares") sourcelang:eng'
    params = {
        "query": q,
        "mode": "artlist",
        "format": "json",
        "maxrecords": 250,
        "startdatetime": ymdhms(start_date),
        "enddatetime": ymdhms(end_date),
        "sort": "datedesc",
    }
    r = requests.get(GDELT_DOC, params=params, timeout=30)
    ct = r.headers.get("content-type")
    print(f"\n{start_date} -> {end_date}")
    print("status:", r.status_code, "| ct:", ct)
    print("head:", r.text[:120].replace("\n"," "))

    # intenta parsear de forma robusta como en tu fetch_block (limpia control chars)
    import json, re
    txt = re.sub(r"[\x00-\x1F\x7F]", "", r.text.strip())
    try:
        j = json.loads(txt)
        print("articles key:", "articles" in j, "| n_articles:", len(j.get("articles", [])))
    except Exception as e:
        print("JSON parse failed:", type(e).__name__)

# bloques “0”
debug_block("2025-11-14", "2025-11-21")
debug_block("2025-11-21", "2025-11-28")
debug_block("2025-11-28", "2025-12-05")

# y el otro que mencionaste
debug_block("2024-12-20", "2024-12-27")



2025-11-14 -> 2025-11-21
status: 200 | ct: application/json; charset=utf-8
head: {"articles": [ { "url": "https://finance.yahoo.com/news/meta-stock-buy-sell-michael-164624263.html", "url_mobile": "", "
articles key: True | n_articles: 250

2025-11-21 -> 2025-11-28
status: 200 | ct: application/json; charset=utf-8
head: {"articles": [ { "url": "https://biztoc.com/x/d658fd0dd79577c0", "url_mobile": "", "title": "Advocacy Group Claims Confl
articles key: True | n_articles: 144

2025-11-28 -> 2025-12-05
status: 200 | ct: application/json; charset=utf-8
head: {"articles": [ { "url": "https://markets.financialcontent.com/stocks/article/marketminute-2025-12-4-gold-shines-bright-r
articles key: True | n_articles: 157

2024-12-20 -> 2024-12-27
status: 200 | ct: application/json; charset=utf-8
head: {"articles": [ { "url": "https://english.news.cn/20241227/46b7ec4b8c72481d884c252cd0db70f6/c.html", "url_mobile": "", "t
JSON parse failed: JSONDecodeError


In [37]:
import pandas as pd

raw = pd.read_csv("AAPL_news_raw.csv")
raw["dt"] = pd.to_datetime(raw["seendate"], errors="coerce", utc=True)
raw = raw.dropna(subset=["dt"])

raw["year"] = raw["dt"].dt.year

print("rows:", raw.shape[0])
print("min/max:", raw["dt"].min().date(), raw["dt"].max().date())
print("\nrows per year:")
print(raw.groupby("year").size().sort_index())

# check extra: si faltan años
years = set(raw["year"].unique())
print("\nmissing among 2022-2026:", [y for y in range(2022, 2027) if y not in years])


rows: 20203
min/max: 2024-02-01 2026-01-21

rows per year:
year
2024    9926
2025    9800
2026     477
dtype: int64

missing among 2022-2026: [2022, 2023]


In [4]:
import os, time, json, re
import pandas as pd
import requests

RAW_OUT = "AAPL_news_raw.csv"
GDELT_DOC = "https://api.gdeltproject.org/api/v2/doc/doc"

def ymdhms(ts):
    return pd.Timestamp(ts).strftime("%Y%m%d%H%M%S")

def fetch_block(start_date, end_date):
    q = '(AAPL OR "Apple stock" OR "Apple shares") sourcelang:eng'
    params = {
        "query": q,
        "mode": "artlist",
        "format": "json",
        "maxrecords": 250,
        "startdatetime": ymdhms(start_date),
        "enddatetime": ymdhms(end_date),
        "sort": "datedesc",
    }
    try:
        r = requests.get(GDELT_DOC, params=params, timeout=30)
    except Exception:
        return pd.DataFrame(columns=["seendate","title","url"])

    if r.status_code != 200:
        return pd.DataFrame(columns=["seendate","title","url"])

    text = re.sub(r"[\x00-\x1F\x7F]", "", r.text.strip())
    s, e = text.find("{"), text.rfind("}") + 1
    if s == -1 or e <= s:
        return pd.DataFrame(columns=["seendate","title","url"])

    try:
        j = json.loads(text[s:e])
    except Exception:
        return pd.DataFrame(columns=["seendate","title","url"])

    arts = j.get("articles", [])
    rows = [{
        "seendate": a.get("seendate"),
        "title": a.get("title"),
        "url": a.get("url"),
    } for a in arts]

    df = pd.DataFrame(rows, columns=["seendate","title","url"])
    df = df.dropna(subset=["seendate","title","url"]).drop_duplicates(subset=["url"])
    return df


def append_raw(df_new):
    if df_new is None or df_new.empty:
        return
    if os.path.exists(RAW_OUT):
        prev = pd.read_csv(RAW_OUT)
        all_df = pd.concat([prev, df_new], ignore_index=True)
    else:
        all_df = df_new.copy()
    all_df = all_df.dropna(subset=["seendate","title","url"]).drop_duplicates(subset=["url"])
    all_df.to_csv(RAW_OUT, index=False)

def download_range(start_date, end_date, sleep_s=2):
    cur, end = pd.Timestamp(start_date), pd.Timestamp(end_date)
    while cur < end:
        nxt = min(cur + pd.Timedelta(days=7), end)

        df = fetch_block(cur, nxt)

        # if it saturates, split into 2-day chunks inside the week
        if len(df) >= 250:
            subs = []
            sub = cur
            while sub < nxt:
                sub_nxt = min(sub + pd.Timedelta(days=2), nxt)
                subs.append(fetch_block(sub, sub_nxt))
                sub = sub_nxt
                time.sleep(sleep_s)
            df = pd.concat(subs, ignore_index=True) if subs else pd.DataFrame()

        append_raw(df)
        print(cur.date(), "->", nxt.date(), "rows:", len(df))

        cur = nxt
        time.sleep(sleep_s)

def resume_from_csv(default_start="2022-01-01"):
    if not os.path.exists(RAW_OUT):
        return pd.Timestamp(default_start)
    raw = pd.read_csv(RAW_OUT)
    dt = pd.to_datetime(raw["seendate"], errors="coerce", utc=True)
    dt = dt.dropna()
    if dt.empty:
        return pd.Timestamp(default_start)
    last = dt.max().date()
    return pd.Timestamp(last) + pd.Timedelta(days=1)

# ---- RUN: 2022 -> hoy, reanudable ----
start = resume_from_csv("2022-01-01")
end = pd.Timestamp.utcnow().strftime("%Y-%m-%d")
print("Resuming from:", start.date(), "to", end)

if start < pd.Timestamp(end):
    download_range(start, end, sleep_s=2)
else:
    print("Nothing new to download.")


Resuming from: 2025-06-15 to 2026-01-21
2025-06-15 -> 2025-06-22 rows: 0
2025-06-22 -> 2025-06-29 rows: 0
2025-06-29 -> 2025-07-06 rows: 88
2025-07-06 -> 2025-07-13 rows: 148
2025-07-13 -> 2025-07-20 rows: 191
2025-07-20 -> 2025-07-27 rows: 118
2025-07-27 -> 2025-08-03 rows: 399
2025-08-03 -> 2025-08-10 rows: 482
2025-08-10 -> 2025-08-17 rows: 199
2025-08-17 -> 2025-08-24 rows: 122
2025-08-24 -> 2025-08-31 rows: 152
2025-08-31 -> 2025-09-07 rows: 261
2025-09-07 -> 2025-09-14 rows: 391
2025-09-14 -> 2025-09-21 rows: 196
2025-09-21 -> 2025-09-28 rows: 229
2025-09-28 -> 2025-10-05 rows: 126
2025-10-05 -> 2025-10-12 rows: 166
2025-10-12 -> 2025-10-19 rows: 170
2025-10-19 -> 2025-10-26 rows: 258
2025-10-26 -> 2025-11-02 rows: 412
2025-11-02 -> 2025-11-09 rows: 212
2025-11-09 -> 2025-11-16 rows: 152
2025-11-16 -> 2025-11-23 rows: 284
2025-11-23 -> 2025-11-30 rows: 145
2025-11-30 -> 2025-12-07 rows: 134
2025-12-07 -> 2025-12-14 rows: 162
2025-12-14 -> 2025-12-21 rows: 137
2025-12-21 -> 2025-1

In [5]:
import pandas as pd

raw = pd.read_csv("AAPL_news_raw.csv")
raw["dt"] = pd.to_datetime(raw["seendate"], errors="coerce", utc=True)

print("rows:", raw.shape[0])
print("min/max:", raw["dt"].min().date(), raw["dt"].max().date())
print(raw.groupby(raw["dt"].dt.year).size())


rows: 48567
min/max: 2022-01-01 2026-01-20
dt
2022    14864
2023    12082
2024    10774
2025    10373
2026      474
dtype: int64


In [10]:
import os, time, json, re, requests
import pandas as pd

RAW_OUT = "AAPL_news_raw.csv"
GDELT_DOC = "https://api.gdeltproject.org/api/v2/doc/doc"

def ymdhms(ts):
    return pd.Timestamp(ts).strftime("%Y%m%d%H%M%S")

# ---------- fetch robusto ----------
def fetch_block(start_date, end_date, tries=4):
    q = '(AAPL OR "Apple stock" OR "Apple shares") sourcelang:eng'
    params = {
        "query": q,
        "mode": "artlist",
        "format": "json",
        "maxrecords": 250,
        "startdatetime": ymdhms(start_date),
        "enddatetime": ymdhms(end_date),
        "sort": "datedesc",
    }

    for k in range(tries):
        try:
            r = requests.get(GDELT_DOC, params=params, timeout=30)
        except Exception:
            time.sleep(2*(2**k)); continue

        if r.status_code != 200:
            time.sleep(2*(2**k)); continue

        text = re.sub(r"[\x00-\x1F\x7F]", "", r.text.strip())
        s, e = text.find("{"), text.rfind("}") + 1
        if s == -1 or e <= s:
            time.sleep(2*(2**k)); continue

        try:
            j = json.loads(text[s:e])
        except Exception:
            time.sleep(2*(2**k)); continue

        arts = j.get("articles", [])
        rows = [{"seendate":a.get("seendate"), "title":a.get("title"), "url":a.get("url")} for a in arts]
        df = pd.DataFrame(rows, columns=["seendate","title","url"])
        df = df.dropna(subset=["seendate","title","url"]).drop_duplicates(subset=["url"])
        return df

    return pd.DataFrame(columns=["seendate","title","url"])

def append_raw(df_new):
    if df_new is None or df_new.empty:
        return
    if os.path.exists(RAW_OUT):
        prev = pd.read_csv(RAW_OUT)
        all_df = pd.concat([prev, df_new], ignore_index=True)
    else:
        all_df = df_new.copy()
    all_df = all_df.dropna(subset=["seendate","title","url"]).drop_duplicates(subset=["url"])
    all_df.to_csv(RAW_OUT, index=False)

# ---------- detectar días faltantes ----------
prices = pd.read_csv("AAPL_processed.csv")
prices["date"] = pd.to_datetime(prices["date"])
prices = prices[prices["date"] >= "2022-01-01"]   # 👈 SOLO desde 2022
trading_days = set(prices["date"].dt.date)


raw = pd.read_csv(RAW_OUT)
raw["dt"] = pd.to_datetime(raw["seendate"], errors="coerce", utc=True)
raw = raw.dropna(subset=["dt"])
raw_days = set(raw["dt"].dt.date)

missing_days = sorted(trading_days - raw_days)

print("Trading days:", len(trading_days))
print("Days with news:", len(raw_days))
print("Missing days:", len(missing_days))
print("Example missing:", missing_days[:10])

# ---------- agrupar días consecutivos ----------
def ranges_from_dates(dates):
    if not dates: return []
    out = []
    start = prev = pd.Timestamp(dates[0])
    for d in dates[1:]:
        dts = pd.Timestamp(d)
        if (dts - prev).days == 1:
            prev = dts
        else:
            out.append((start, prev + pd.Timedelta(days=1)))
            start = prev = dts
    out.append((start, prev + pd.Timedelta(days=1)))
    return out

ranges = ranges_from_dates(missing_days)
print("Ranges to fill:", len(ranges))

# ---------- descargar solo esos rangos ----------
for a, b in ranges:
    cur = a
    while cur < b:
        nxt = min(cur + pd.Timedelta(days=7), b)
        df = fetch_block(cur, nxt)

        if len(df) >= 250:
            subs = []
            sub = cur
            while sub < nxt:
                sub_nxt = min(sub + pd.Timedelta(days=2), nxt)
                subs.append(fetch_block(sub, sub_nxt))
                sub = sub_nxt
                time.sleep(2)
            df = pd.concat(subs, ignore_index=True) if subs else pd.DataFrame(columns=["seendate","title","url"])

        print("Fill:", cur.date(), "->", nxt.date(), "rows:", len(df))
        append_raw(df)
        cur = nxt
        time.sleep(2)

print("Done filling gaps.")


Trading days: 1014
Days with news: 1447
Missing days: 14
Example missing: [datetime.date(2023, 3, 23), datetime.date(2024, 12, 23), datetime.date(2024, 12, 24), datetime.date(2025, 6, 16), datetime.date(2025, 6, 17), datetime.date(2025, 6, 18), datetime.date(2025, 6, 20), datetime.date(2025, 6, 23), datetime.date(2025, 6, 24), datetime.date(2025, 6, 25)]
Ranges to fill: 6
Fill: 2023-03-23 -> 2023-03-24 rows: 0
Fill: 2024-12-23 -> 2024-12-25 rows: 0
Fill: 2025-06-16 -> 2025-06-19 rows: 0
Fill: 2025-06-20 -> 2025-06-21 rows: 0
Fill: 2025-06-23 -> 2025-06-28 rows: 0
Fill: 2025-06-30 -> 2025-07-02 rows: 0
Done filling gaps.


In [11]:
import pandas as pd, numpy as np, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# -------- CONFIG --------
RAW_FILE = "AAPL_news_raw.csv"
OUT_FILE = "AAPL_sentiment_daily.csv"
BATCH = 64

# -------- DEVICE --------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# -------- LOAD FINBERT --------
model_name = "ProsusAI/finbert"
tok = AutoTokenizer.from_pretrained(model_name)
mdl = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
mdl.eval()

# -------- LOAD RAW --------
raw = pd.read_csv(RAW_FILE)
raw = raw.dropna(subset=["title", "seendate", "url"]).drop_duplicates(subset=["url"])

raw["dt"] = pd.to_datetime(raw["seendate"], errors="coerce", utc=True)
raw["date"] = raw["dt"].dt.date
raw = raw.dropna(subset=["date"]).reset_index(drop=True)

print("RAW rows:", raw.shape)
print("Date range:", raw["date"].min(), "->", raw["date"].max())

# -------- FINBERT INFERENCE --------
scores = []

for i in range(0, len(raw), BATCH):
    texts = raw["title"].iloc[i:i+BATCH].tolist()

    enc = tok(texts, padding=True, truncation=True, max_length=64, return_tensors="pt")
    enc = {k: v.to(device) for k, v in enc.items()}

    with torch.no_grad():
        probs = torch.softmax(mdl(**enc).logits, dim=-1).cpu().numpy()

    scores.append(probs[:, 2] - probs[:, 0])  # pos - neg

raw["sent_score"] = np.concatenate(scores)

# -------- AGREGAR POR DÍA --------
daily = raw.groupby("date").agg(
    sent_mean=("sent_score", "mean"),
    sent_std=("sent_score", "std"),
    sent_volume=("sent_score", "size")
).reset_index()

daily["sent_std"] = daily["sent_std"].fillna(0.0)
daily = daily.sort_values("date").reset_index(drop=True)

# -------- FEATURES TEMPORALES --------
daily["sent_mean_3d"] = daily["sent_mean"].rolling(3, min_periods=1).mean()
daily["sent_mean_5d"] = daily["sent_mean"].rolling(5, min_periods=1).mean()
daily["sent_momentum"] = daily["sent_mean"] - daily["sent_mean"].shift(1)

# -------- SAVE --------
daily.to_csv(OUT_FILE, index=False)

print("\nSaved:", OUT_FILE)
print("Rows:", daily.shape)
print("Date range:", daily["date"].min(), "->", daily["date"].max())
print(daily.head())


device: cuda
RAW rows: (50022, 5)
Date range: 2022-01-01 -> 2026-01-20

Saved: AAPL_sentiment_daily.csv
Rows: (1447, 7)
Date range: 2022-01-01 -> 2026-01-20
         date  sent_mean  sent_std  sent_volume  sent_mean_3d  sent_mean_5d  \
0  2022-01-01   0.743096  0.379356           37      0.743096      0.743096   
1  2022-01-02   0.751415  0.315719           16      0.747256      0.747256   
2  2022-01-03   0.817959  0.000000            1      0.770824      0.770824   
3  2022-01-04  -0.083785  0.702841          250      0.495196      0.557171   
4  2022-01-05   0.311352  0.459763           62      0.348509      0.508007   

   sent_momentum  
0            NaN  
1       0.008319  
2       0.066545  
3      -0.901744  
4       0.395137  


In [12]:
from google.colab import files
files.download("AAPL_news_raw.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [17]:
import pandas as pd

# Cargar el archivo
df = pd.read_csv("AAPL_sentiment_daily.csv")

# Ver las primeras filas
df.head()




,date,sent_mean,sent_std,sent_volume,sent_mean_3d,sent_mean_5d,sent_momentum
0,2022-01-01,0.743096,0.379356,37,0.743096,0.743096,NaN
1,2022-01-02,0.751415,0.315719,16,0.747256,0.747256,0.008319
2,2022-01-03,0.817959,0.000000,1,0.770824,0.770824,0.066545
3,2022-01-04,-0.083785,0.702841,250,0.495196,0.557171,-0.901744
4,2022-01-05,0.311352,0.459763,62,0.348509,0.508007,0.395137


In [18]:
import pandas as pd

# Cargar datos
df = pd.read_csv("AAPL_sentiment_daily.csv")

# Asegurar fecha como datetime y ordenar
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

# Establecer índice temporal
df = df.set_index("date")

# -----------------------------
# 1. Imputar variables base
# -----------------------------

# sent_volume: si no hay noticias → 0
df["sent_volume"] = df["sent_volume"].fillna(0)

# sent_mean y sent_std: interpolación temporal + ffill/bfill
for col in ["sent_mean", "sent_std"]:
    df[col] = df[col].interpolate(method="time")
    df[col] = df[col].ffill().bfill()

# -----------------------------
# 2. Recalcular variables derivadas (mejor que imputarlas)
# -----------------------------

df["sent_mean_3d"] = df["sent_mean"].rolling(window=3, min_periods=1).mean()
df["sent_mean_5d"] = df["sent_mean"].rolling(window=5, min_periods=1).mean()

# Momentum: diferencia día a día
df["sent_momentum"] = df["sent_mean"].diff()

# -----------------------------
# 3. Reset índice si quieres guardar
# -----------------------------
df = df.reset_index()

# Guardar resultado
df.to_csv("AAPL_sentiment_filled.csv", index=False)

print("Imputación completa y features recalculadas.")

Imputación completa y features recalculadas.


De este notebook obtenemos:
- AAPL_sentiment_daily.csv
- AAPL_news_raw.csv

In [19]:
from google.colab import files
files.download("AAPL_sentiment_daily.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>